In [1]:
! pip install openai requests -q

# Setup and First API Call

In [2]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

In [3]:
# Load the open router api key
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [4]:
# Create a client
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])


In [7]:
# Define models to use
FREE_MODEL = 'openai/gpt-oss-120b:free'
TOOL_MODEL = 'openai/gpt-oss-120b:free'

## First API call
We send tokens, get tokens back

In [10]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[
        {
            "role": "user",
            "content": "What is 2+2? Answer in one word"
        }
    ],
    temperature = 0.0,
    max_tokens = 50
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens:\nIn: {response.usage.prompt_tokens}\nOut: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Four
Tokens:
In: 78
Out: 16
Model used: openai/gpt-oss-120b:free


## Trying out various temperatures

In [15]:
prompt = 'Write a one sentence story about a robot'

# for temperatures 0, 1 and 2
for temp in [0.0, 1.0, 2.0, 3.0]:
  results = []

  # get 3 responses for each temperature
  for _ in range(3):
    response = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature = temp,
        max_tokens = 60
    )
    # Extract the response message
    content = response.choices[0].message.content

    # Append to results
    if content:
      results.append(content.strip())
    else:
      results.append('No content returned')

  # Print all 3 responses for current temperature
  print(f'Temperature: {temp}')

  for i, text in enumerate(results):
    print(f'{i + 1}: {text}')

  print("\n")

Temperature: 0.0
1: The lone robot, powered by a heart‑shaped battery, trekked across the silent desert to deliver a single, handwritten love letter to the last human still dreaming.
2: A lone robot, powered by the echo of forgotten lullabies, trekked across the silent desert to plant a single, humming seed that would one day bloom into a chorus of light.
3: A lone robot, powered by the echo of forgotten lullabies, trekked across the silent desert to deliver a single, handwritten note to the last human who still dreamed of sunrise.


Temperature: 1.0
1: A lone robot, powered by forgotten solar panels, trekked across the rust‑colored desert to deliver a single, handwritten postcard to the last human who still remembered the sound of rain.
2: A lone robot, powered by forgotten dreams, whispered the sunrise into a world that had stopped listening.
3: The lonely robot, programmed to garden, planted a single chrome seed in the cracked desert and watched, trembling with borrowed hope, as a m

BadRequestError: Error code: 400 - {'error': {'message': 'Expected temperature to be at most 2, received 3', 'code': 400, 'metadata': {'provider_name': None}}, 'user_id': 'user_3ErmpVfOt2yuz5KDcc3Wtful5ps'}

## The model has NO memory

Multi-turn: The entire conversation is sent everytime, "Chat memory is our application sending the entire chat history"

In [21]:
conversation = [
    {
        'role': 'system',
        'content': 'You are a math tutor. Be concise'
    },
    {
        'role': 'user',
        'content': 'What is a dervative?'
    }
]

In [22]:
r1 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    temperature = 0.0,
    max_tokens = 60
)

turn1 = r1.choices[0].message.content
print(f'Turn 1: {turn1}')
print(f'Prompt tokens: {r1.usage.prompt_tokens}')
print(f'Completion tokens: {r1.usage.completion_tokens}')

Turn 1: A **derivative** measures how a function changes as its input changes.  

For a function \(f(x)\), the derivative at a point \(x\) is  

\[
f'(x)=\lim_{h\to 0}\frac{f(x+h)-f(x)}{h},
\]

the instantaneous rate of change (slope of the tangent line) at that point.  

In physics, \(f'(x)\) often represents velocity (rate of change of position), acceleration (rate of change of velocity), etc.  

Key ideas:
- **Notation:** \(f'(x),\; \frac{df}{dx},\; Df(x)\)
- **Interpretation:** “how fast” \(f\) is changing with respect to \(x\)
- **Computation:** use rules (power, product, quotient, chain) after finding the limit definition.
Prompt tokens: 86
Completion tokens: 212


In [24]:
# append the previous chat and the new question to conversation list
conversation.append(
    {
        "role": "assistant",
        "content": turn1
    }
)
conversation.append(
        {
        "role": "user",
        "content": "Demonstrate on x^2"
    }
)

In [25]:
r2 = client.chat.completions.create(
    model = FREE_MODEL,
    messages = conversation,
    temperature = 0.0,
    max_tokens = 60
)

turn2 = r2.choices[0].message.content
print(f'Turn 2: {turn2}')
print(f'Prompt tokens: {r2.usage.prompt_tokens}')
print(f'Completion tokens: {r2.usage.completion_tokens}')

Turn 2: **Derivative of \(f(x)=x^{2}\) using the limit definition**

\[
f'(x)=\lim_{h\to 0}\frac{f(x+h)-f(x)}{h}
      =\lim_{h\to 0}\frac{(x+h)^{2}-x^{2}}{h}.
\]

1. Expand \((x+h)^{2}\):

\[
(x+h)^{2}=x^{2}+2xh+h^{2}.
\]

2. Substitute and simplify the numerator:

\[
\frac{(x^{2}+2xh+h^{2})-x^{2}}{h}
   =\frac{2xh+h^{2}}{h}
   =\frac{h(2x+h)}{h}=2x+h.
\]

3. Take the limit as \(h\to 0\):

\[
f'(x)=\lim_{h\to 0}(2x+h)=2x.
\]

---

### Quick check with the power rule
The power rule says \(\displaystyle \frac{d}{dx}x^{n}=n x^{\,n-1}\).  
For \(n=2\): \(\displaystyle \frac{d}{dx}x^{2}=2x^{1}=2x\).

Both methods give the same result:  

\[
\boxed{\,\frac{d}{dx}x^{2}=2x\,}.
\]
Prompt tokens: 287
Completion tokens: 330


# Function calling: Bridge to agents

- Function calling = The Model can <b>REQUEST</b> tools. It outputs structured JSON saying "I want to call X with args Y".
- <b>YOUR CODE</b> executes the function and feeds the result back to the model.
- The model never <b>"calls"</b> anything. It generates text that says what it wants to happen.

## Define Tools

We describe tools so that the model knows whats available. The model <b>does not Have</b> these tools. It can only <b> REQUEST </b> them.

In [26]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature and conditions",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city. for example, Tokyo"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression. for example, (5 + 3) * 2"
                    }
                },
                "required": ["expression"]
            }
        }
    },
]

## The model decides to use a tool

In [27]:
r = client.chat.completions.create(
    model = FREE_MODEL,
    messages = [{"role": "user", "content": "What is the weather in Tokyo?"}],
    tools = tools,
    tool_choice = "auto",
    temperature = 0.0
)

msg = r.choices[0].message
print(f"Finish reason: {r.choices[0].finish_reason}")
print(f"Message: {msg.content}")
print(f'Tool calls: {msg.tool_calls}')

if msg.tool_calls:
  tc = msg.tool_calls[0]
  print(f"The model wants to call: {tc.function.name}({tc.function.arguments})")
  print(f"The model GENERATED a JSON requesting a function. Our code must execute that function and return the result")

Finish reason: tool_calls
Message: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-bfd0666773d21aec', function=Function(arguments='{\n  "city": "Tokyo"\n}', name='get_weather'), type='function', index=0)]
The model wants to call: get_weather({
  "city": "Tokyo"
})
The model GENERATED a JSON requesting a function. Our code must execute that function and return the result


## Complete the tool call loop

User Prompt -> Model requests the tool -> User code runs the requested tool -> Returns the result to model -> Final response from the model

In [31]:
# Dummy tool implementation
def get_weather(city):
  fake = {
      "Tokyo": {"temp": "22 C", "condition": "Clear"},
      "New Delhi": {"temp": "42 C", "condition": "Sunny"},
      "London": {"temp": "21 C", "condition": "Cloudy"}
  }

  return json.dumps(fake.get(city, {"temp": "Unknown", "condition": "Unknown"}))

def calculate(expression):
    try:
      allowed = set("0123456789+-*/.() ")

      if not all(c in allowed for c in expression):
        return json.dumps({"error": "Invalid characters"})

      return json.dumps({"result": eval(expression)})

    except Exception as e:
      return json.dumps({"error": str(e)})

In [32]:
TOOL_FUNCS = {"get_weather": get_weather, "calculate": calculate}

In [33]:
# Execute the tool call

if msg.tool_calls:
  tc = msg.tool_calls[0]
  fn = TOOL_FUNCS[tc.function.name]
  args = json.loads(tc.function.arguments)
  result = fn(**args)
  print(f"tool results: {result}")

  # Feed the result back to the model
  messages = [
      {"role": "user", "content": "What is the weather in Tokyo?"},
      msg,
      {
          "role": "tool",
          "tool_call_id": tc.id,
          "content": result
      }
  ]

  r2 = client.chat.completions.create(
      model = FREE_MODEL,
      messages = messages,
      tools = tools,
      temperature = 0.0
  )

  print(f"Final answer: {r2.choices[0].message.content}")
  print("\n→ Full cycle: user → model requests tool → we run it → model answers")

tool results: {"temp": "22 C", "condition": "Clear"}
Final answer: The current weather in Tokyo is **22 °C** with clear skies.

→ Full cycle: user → model requests tool → we run it → model answers


---
# Part 3: Build a ReAct Agent From Scratch

**No LangChain. No CrewAI. No frameworks.**

Just Python + an API + a loop.

The **ReAct** pattern: `THINK → ACT → OBSERVE → repeat until done`

## The agent's system prompt

- This prompt <b>IS</b> the architecture. It tells the model to alternate between <b>THOUGHT</b> and <b> ACTION </b> in a parseable format.

In [34]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to these tools:
{tool_descriptions}

## How to respond

When you need to use a tool, respond in EXACTLY this format:

THOUGHT: <your reasoning about what to do next>
ACTION: <tool_name>
ACTION_INPUT: <arguments as valid JSON>

When you have enough information for the final answer:

THOUGHT: <your final reasoning>
FINAL_ANSWER: <your complete answer to the user>

## Rules
- Always start with THOUGHT
- Use only ONE action per turn
- Wait for the OBSERVATION before continuing
- If a tool returns an error, reason about it and try a different approach
- Be concise in your thoughts
"""

## Real tools

Wikipedia search hits a real API. The calculator does real math. These are the "hands" of our agent.

In [35]:
def search_wikipedia(query):
    """Search Wikipedia and return a summary."""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return json.dumps({
                "title": data.get("title", ""),
                "summary": data.get("extract", "No summary found.")[:800]
            })
        return json.dumps({"error": f"Page not found for '{query}'. Try a different term."})
    except Exception as e:
        return json.dumps({"error": str(e)})

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

# Tool registry
TOOLS = {
    "search_wikipedia": {
        "fn": search_wikipedia,
        "desc": "search_wikipedia(query: str) — Search Wikipedia. Use simple topic names like 'France' or 'Albert Einstein'."
    },
    "calculate": {
        "fn": calculate_math,
        "desc": "calculate(expression: str) — Evaluate a math expression. Example: '(5 + 3) * 2.5'"
    },
    "get_current_date": {
        "fn": get_current_date,
        "desc": "get_current_date() — Get today's date and time. Pass empty JSON: {}"
    },
}

print(f"Tools registered: {list(TOOLS.keys())}")

Tools registered: ['search_wikipedia', 'calculate', 'get_current_date']


## 🔥 THE AGENT LOOP

This is the heart of the class. **~60 lines** that implement the same pattern as Claude Code and Codex.

Walk through it:
1. Build system prompt with tool descriptions
2. Enter a loop (max_iterations prevents infinite loops)
3. Each iteration: call the LLM, get text with THOUGHT + ACTION or FINAL_ANSWER
4. If FINAL_ANSWER → done
5. If ACTION → parse tool name + args, execute the tool
6. Inject OBSERVATION back as a user message
7. Loop

In [37]:
def run_agent(user_query, max_iterations=10, verbose=True):
    """
    A ReAct agent from scratch.
    No frameworks. Just an LLM + tools + a loop.
    """
    # Build system prompt with tool descriptions
    tool_desc = "\n".join(f"- {t['desc']}" for t in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1}/{max_iterations} ---")

        # STEP 1: Ask the LLM what to do
        response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=messages,
            temperature=0,
            max_tokens=800,
        )

        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        # STEP 2: Parse the response
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)", text, re.DOTALL
        )
        thought = thought_match.group(1).strip() if thought_match else ""

        if verbose and thought:
            print(f"💭 THOUGHT: {thought[:200]}")

        # Check for FINAL_ANSWER
        if "FINAL_ANSWER:" in text:
            answer = text.split("FINAL_ANSWER:")[-1].strip()
            if verbose:
                print(f"\n{'='*60}")
                print(f"✅ AGENT FINISHED in {i+1} iteration(s)")
                print(f"{'='*60}")
                print(f"📝 ANSWER: {answer[:500]}")
            return {"answer": answer, "iterations": i + 1}

        # Check for ACTION
        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\n|$)", text, re.DOTALL)

        if not action_match:
            messages.append({
                "role": "user",
                "content": "Please respond with either ACTION + ACTION_INPUT or FINAL_ANSWER."
            })
            if verbose:
                print("⚠️  Format issue — nudging...")
            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # STEP 3: Execute the tool
        if tool_name not in TOOLS:
            observation = json.dumps({
                "error": f"Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"
            })
        else:
            try:
                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}
                observation = TOOLS[tool_name]["fn"](**args)
            except Exception as e:
                observation = json.dumps({"error": f"Failed to call {tool_name}: {e}"})

        if verbose:
            print(f"🔧 ACTION: {tool_name}({raw_input[:80]})")
            obs_preview = observation[:200] + "..." if len(observation) > 200 else observation
            print(f"👁️  OBSERVATION: {obs_preview}")

        # STEP 4: Feed observation back
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    if verbose:
        print(f"\n⚠️  Max iterations ({max_iterations}) reached.")
    return {"answer": "Max iterations reached.", "iterations": max_iterations}

### Test — Simple question

Watch the loop: `THOUGHT → ACTION → OBSERVATION → FINAL_ANSWER`

In [39]:
result = run_agent("What is 123 * 123?")


🧑 USER: What is 123 * 123?

--- Iteration 1/10 ---
💭 THOUGHT: Use calculate tool.
⚠️  Format issue — nudging...

--- Iteration 2/10 ---
💭 THOUGHT: Need to compute 123 * 123. Use calculate.
🔧 ACTION: calculate({"expression": "123 * 123"})
👁️  OBSERVATION: {"expression": "123 * 123", "result": 15129}

--- Iteration 3/10 ---
💭 THOUGHT: The calculation result is 15129.

✅ AGENT FINISHED in 3 iteration(s)
📝 ANSWER: 123 multiplied by 123 equals 15,129.


In [41]:
result = run_agent(
    "Search Wikipedia for 'Wakanda' and tell me about its history."
)


🧑 USER: Search Wikipedia for 'Wakanda' and tell me about its history.

--- Iteration 1/10 ---
💭 THOUGHT: Use search_wikipedia for 'Wakanda'.

✅ AGENT FINISHED in 1 iteration(s)
📝 ANSWER: Wakanda is a fictional African nation created by Marvel Comics, first appearing in *Fantastic Four* #52 (1966). According to its canon history:

- **Founding**: The story begins when a massive meteorite of the rare metal **vibranium** crashes in a remote part of Africa. The first ruler, **Bashenga**, becomes the inaugural **Black Panther** and establishes Wakanda as a unified kingdom, dedicating the nation to protecting the vibranium and developing its unique technology in secrecy.

- **Isolatio
